In [ ]:
import utils.IV as iv

iv.main(
    options_data_path="Option_Data.xlsx",
    INPUT_DIR=".",
    OUTPUT_DIR="IV_Out",
    k_window=0.15,
)


In [1]:
import utils.Event_Detection as event_det

event_det.main(
    INPUT_DIR="IV_Out",
    INPUT_FILE="cleaned_quotes_with_iv_1dte.csv",
    OUTPUT_DIR="Event_Det_Out",
)


[INFO] Loaded rows: 508,896
[INFO] Time range: 2026-01-22 09:30:00 to 2026-03-05 16:14:00
[INFO] Cross-sections to process: 9,766


KeyboardInterrupt: 

In [2]:
import importlib

import utils.Butterfly as butterfly

butterfly.main(INPUT_DIR="IV_Out", OUTPUT_DIR="Butterfly_Out")


Top day summary:
trade_date  n_obs  n_events  max_event_score  sigma_range  skew_range  curvature_range
2026-01-28    391        10        20.658792     0.028444    2.663329       683.819564
2026-02-03    391        15        17.875404     0.101012    3.349172       741.381739
2026-01-27    391        17        17.296564     0.018640    2.117571       346.890606
2026-02-04    391        16        16.719057     0.081220    3.911756       731.387836
2026-02-18    391        21        16.589134     0.086372    2.527142       534.712690
2026-02-23    391        26        16.476229     0.058461    2.422420       491.205325
2026-02-17    391        17        16.063009     0.086561    4.010358       369.715857
2026-02-11    391        22        15.522630     0.072327    2.436755       528.939727
2026-02-25    391        10        15.085869     0.019076    1.356256       320.171019
2026-01-26    391        19        14.476857     0.038483    3.089212       484.171483

Selected intraday day: 20

In [3]:
import utils.Regime_Label as regime

regime.main(
    INPUT_DIR="Butterfly_Out",
    OUTPUT_DIR="Regime_Out",
    features_file="option_b_intraday_features_events_1dte.csv",
)


Regime fit summary:
       feature  regime regime_name  n_obs     alpha     beta        mu  lambda_per_min  half_life_min  shock_std       r2
curvature_bfly       0        slow   5372  6.690288 0.900813 67.451273        0.104458       6.635682  66.534909 0.796691
curvature_bfly       1        fast   3988  6.710755 0.808454 35.034699        0.212631       3.259853  80.937920 0.667707
     sigma_atm       0        slow   5372  0.000277 0.998388  0.171955        0.001614     429.514817   0.002507 0.996912
     sigma_atm       1        fast   3988  0.000396 0.997179  0.140354        0.002825     245.346142   0.003231 0.995636
          skew       0        slow   5372 -0.494040 0.849764 -3.288435        0.162796       4.257759   0.332181 0.702465
          skew       1        fast   3988 -0.794297 0.757411 -3.274246        0.277850       2.494685   0.445355 0.583001

Fast vs slow comparison:
       feature  slow_lambda_per_min  fast_lambda_per_min  lambda_ratio_fast_to_slow  slow_half_life_

In [4]:
# 2) Model calibration
import utils.Model_Calibration as calib

calib.main(INPUT_DIR="Regime_Out", OUTPUT_DIR="Calibration_Out")


Loading features from Regime_Out/regime_labeled_intraday_features_1dte.csv...
Selecting optimal shock half-life (phi)...
Fitting reduced-form model on train + validation sets...
Generating feature predictions for test set...
Exporting calibration data...

Model Calibration Complete
Chosen shock-state phi_y: 0.9170

Calibrated Parameters Snapshot:
        component regime_name  mean_reversion_per_min  shock_loading_beta_y
0           v_atm        slow                0.001571              0.000107
1           v_atm        fast                0.003363             -0.000020
2            skew        slow                0.158028              0.017371
3            skew        fast                0.246253              0.012203
4  curvature_bfly        slow                0.087383             -8.836286

Predictions saved. You are ready to run BackTestFinal.py.


In [1]:
import utils.BackTestFinal as bt

bt.main(
    IV_INPUT_DIR="IV_Out",
    FEATURE_INPUT_DIR="Calibration_Out",
    OUTPUT_DIR="Backtest_Out",
    stock_tc_per_share=0.02,
    option_tc_per_unit=0.10,
)


[DEBUG] feature rows: 1560 | feature dates: 4
[DEBUG] quote transition rows: 72360 | quote dates: 4
[DEBUG] hedge choice rows: 8
[DEBUG] main panel rows: 69666
Overall summary:
                 method  n_obs      mae     rmse  p95_abs_err  mean_err  n_rehedges  stock_turnover  option_turnover   total_tc  avg_tc_per_day  avg_rehedges_per_day
deltavega_const_time_bs  69666 0.057211 0.111443     0.228159 -0.001838       69663      736.397261      1607.314608 175.459406       43.864852              17415.75
    deltavega_regime_bs  69666 0.057219 0.111435     0.227748 -0.001454       38218      605.995888      1364.930906 148.613008       37.153252               9554.50
 deltavega_regime_model  69666 0.057526 0.111486     0.228516 -0.001718       38969      781.712121      1528.062834 168.440526       42.110131               9742.25

Reduction vs constant-time rehedge:
                method  mae_ratio_to_constant  rmse_ratio_to_constant  rehedge_ratio_to_constant  stock_turnover_ratio_to_